# ISS decoding with Bardensr

This notebook runs Bardensr as a joint spot detector and barcode decoder. It uses the same registration, white-top-hat filtering, and channel normalization as the standard Starfish workflow, but bypasses both BlobDetector/Spotiflow and the Starfish/PoSTcode decoders.

SpaceTx is the common input format. If it already exists, leave `CREATE_SPACETX = False`. Start with one representative region and inspect the evidence distribution and spatial calls before launching the full experiment.

## Environment

From the repository root, install the project and decoder integrations with:

```bash
python -m pip install -U "./ISS_decoding[postcode,spotiflow,istdeco,bardensr]"
```

On Linux the Bardensr extra installs TensorFlow's matching CUDA runtime libraries. This does not require the system CUDA toolkit version to match PyTorch's wheel.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import bardensr

from ISS_decoding import SpaceTx_format as STX
from ISS_decoding import decoding as DEC

print("TensorFlow:", tf.__version__)
print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))
print("Bardensr:", bardensr.__version__)

## Paths and experiment layout

`EXPERIMENT_ROOT` must contain `R1`, `R2`, ... directories. With no separate output root, SpaceTx and decoded outputs are written below each region's `decoding/` directory.

In [ ]:
EXPERIMENT_ROOT = Path("/mnt/DATA/path/to/experiment")
CODEBOOK_CSV = Path("/mnt/DATA/path/to/codebook.csv")
OUTPUT_ROOT = None  # or Path("/mnt/DATA/path/to/decoding_outputs")
REGIONS_TO_PROCESS = [1]

CREATE_SPACETX = False
RUN_DECODING = False

PIXEL_TO_UM = 1.0
CHANNELS = ["DAPI", "Cy3", "Cy5", "AF750", "AF488"]
DECODING_CHANNELS = ["AF750", "Cy5", "Cy3", "AF488"]
NUCLEI_CHANNEL = "DAPI"
USE_CARE_IMAGES = False

## Optional: create SpaceTx

Run this only when `experiment.json` and `codebook.json` have not already been generated. Bardensr does not need a second image or codebook format: the adapter reads both directly from SpaceTx.

In [ ]:
if CREATE_SPACETX:
    STX.make_spacetx_format(
        input_dir=EXPERIMENT_ROOT,
        codebook_csv=CODEBOOK_CSV,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=OUTPUT_ROOT,
        pixel_to_um=PIXEL_TO_UM,
        channels=CHANNELS,
        DO_decorators=DECODING_CHANNELS,
        nuclei_channel=NUCLEI_CHANNEL,
        CARE=USE_CARE_IMAGES,
    )
else:
    print("SpaceTx creation is disabled; existing SpaceTx files will be used.")

## Bardensr settings

Start with `singleshot`: it is fast and returns a barcode-correlation evidence score. The default `0.72` threshold follows Bardensr's example. `normalize_frames=True` makes the noise floor and threshold comparable across imaging frames.

`tile_size` bounds GPU memory. `overlap=None` chooses a peak/PSF-safe halo automatically, and only non-overlapping tile cores are retained to avoid duplicate seam calls.

In [ ]:
BARDENSR_KWARGS = {
    "method": "singleshot",
    "noisefloor": 0.05,
    "peak_threshold": 0.72,
    "peak_threshold_fraction": None,
    "poolsize": (1, 1, 1),
    "tile_size": (512, 512),
    "overlap": None,
    "normalize_frames": True,
    "device": "auto",
    "z_projection": "max",
}

PIPELINE_KWARGS = {
    "register": False,
    "register_dapi": False,
    "masking_radius": 15,
    "normalization_method": "MH",
}

### Optional iterative method

The iterative method estimates a density model and per-frame gains. It is much slower. Its example threshold is relative to each internal tile's maximum; after a pilot, an absolute threshold may be preferable for uniform filtering across tiles.

In [ ]:
# Uncomment to use the optimization-based decoder.
# BARDENSR_KWARGS.update({
#     "method": "iterative",
#     "peak_threshold": None,
#     "peak_threshold_fraction": 0.1,
#     "l1_penalty": 0.0,
#     "psf_radius": (0, 1, 1),
#     "iterations": 100,
#     "estimate_codebook_gain": True,
#     "estimate_colormixing": False,
# })

In [ ]:
if RUN_DECODING:
    DEC.process_experiment(
        input_dir=EXPERIMENT_ROOT,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=OUTPUT_ROOT,
        decode_mode="BARDENSR",
        bardensr_kwargs=BARDENSR_KWARGS,
        **PIPELINE_KWARGS,
    )
else:
    print("Decoding is disabled. Set RUN_DECODING = True when ready.")

## Inspect the result

Parquet is canonical; CSV is written alongside it for compatibility. `bardensr_evidence` is a score, not a calibrated probability. Adjust `RESULT_REGION` if you processed another region.

In [ ]:
RESULT_REGION = "R1"
base = OUTPUT_ROOT if OUTPUT_ROOT is not None else EXPERIMENT_ROOT
result_dir = base / RESULT_REGION / "decoding" / "2_decoded_bardensr"
result_path = result_dir / f"{RESULT_REGION}_decoded_bardensr.parquet"

if result_path.exists():
    decoded = pd.read_parquet(result_path)
    display(decoded.head())
    print(f"{len(decoded):,} accepted transcripts")
    print(decoded["target"].value_counts().head(20))
else:
    print("No result yet:", result_path)

In [ ]:
if result_path.exists() and len(decoded):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].hist(decoded["bardensr_evidence"], bins=80)
    axes[0].set(title="Accepted evidence", xlabel="Bardensr evidence")
    ratio = decoded["bardensr_evidence"] / decoded["bardensr_peak_threshold"]
    axes[1].hist(ratio, bins=80)
    axes[1].set(title="Evidence / threshold", xlabel="threshold multiple")
    axes[2].scatter(decoded["x"], decoded["y"], s=1, alpha=0.5)
    axes[2].invert_yaxis()
    axes[2].set(title="Decoded spot positions", xlabel="x (pixels)", ylabel="y (pixels)")
    plt.tight_layout()

## Reproducibility

Each productive run writes XML and JSON manifests containing the effective Bardensr settings, installed version, pinned fork commit, coordinate units, and output paths. Per-FOV Parquet files in `tiles/` are restart checkpoints; delete only the specific checkpoint(s) you intentionally want to recompute. Dense target-density maps are intentionally not retained because their storage cost is very large.

In [ ]:
manifests = sorted(result_dir.glob("decoding_run_*.json")) if result_dir.exists() else []
if manifests:
    manifest = json.loads(manifests[-1].read_text())
    print(json.dumps(manifest, indent=2))
else:
    print("No Bardensr run manifest found.")